# Experimento 013 - YOLO26 completo via TAO + Ultralytics

O cientista fornece uma pasta YOLO local. Este notebook valida o dataset, envia os arquivos ao SeaweedFS, registra o dataset no TAO e submete o treinamento YOLO26 ao adapter Ultralytics. Nenhum `aws`, `curl` ou upload manual é necessário.

In [1]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "boto3", "pyyaml", "requests"])

0

In [2]:
import json
import os
import time
import uuid
from pathlib import Path

import boto3
import requests
import yaml

TAO_BASE_URL = os.getenv("TAO_BASE_URL", "http://100.107.81.126:8090").rstrip("/")
ADAPTER_URL = os.getenv("TAO_ULTRALYTICS_URL", "http://100.107.81.126:9200").rstrip("/")
S3_ENDPOINT = os.getenv("TAO_SEAWEEDFS_PUBLIC_ENDPOINT", "http://100.107.81.126:18333")
S3_BUCKET = os.getenv("TAO_SEAWEEDFS_BUCKET", "tao-storage")
S3_ACCESS_KEY = os.getenv("TAO_SEAWEEDFS_ACCESS_KEY", "seaweedfs")
S3_SECRET_KEY = os.getenv("TAO_SEAWEEDFS_SECRET_KEY", "seaweedfs123")
TAO_ORG = os.getenv("TAO_ORG", "getter")
WORKSPACE_ID = os.getenv("TAO_TEST_WORKSPACE_ID", "98c4bec1-88df-423e-8a15-af78881e9742")
LOCAL_DATASET_DIR = Path(os.getenv("YOLO_DATASET_DIR", "/tmp/yolo26-smoke")).expanduser().resolve()
DATASET_NAME = os.getenv("YOLO_DATASET_NAME", f"yolo26-scientist-{uuid.uuid4().hex[:8]}")
S3_PREFIX = f"data/{DATASET_NAME}"
RUN_EXPERIMENT = os.getenv("RUN_EXPERIMENT_013", "true").lower() == "true"
POLL_SECONDS = int(os.getenv("YOLO_JOB_POLL_SECONDS", "10"))
TIMEOUT_SECONDS = int(os.getenv("YOLO_JOB_TIMEOUT_SECONDS", "3600"))
configured = os.getenv("TAO_SECRETS_FILE", "")
candidates = [Path(configured)] if configured else []
candidates += [Path("secrets.json"), Path.cwd() / "secrets.json", Path.cwd().parent / "secrets.json", Path.cwd().parent.parent / "secrets.json"]
secrets_file = next((path for path in candidates if path.exists()), None)
NGC_KEY = json.loads(secrets_file.read_text(encoding="utf-8"))["ngc_api_key"] if secrets_file else os.getenv("NGC_KEY", "")
if not NGC_KEY:
    raise RuntimeError("NGC key not found in secrets.json or TAO_SECRETS_FILE")
http = requests.Session()
print({"tao": TAO_BASE_URL, "adapter": ADAPTER_URL, "s3": S3_ENDPOINT, "dataset": str(LOCAL_DATASET_DIR), "run": RUN_EXPERIMENT})

{'tao': 'http://100.107.81.126:8090', 'adapter': 'http://100.107.81.126:9200', 's3': 'http://100.107.81.126:18333', 'dataset': '/tmp/yolo26-smoke', 'run': True}


In [3]:
data_yaml = LOCAL_DATASET_DIR / "data.yaml"
if not data_yaml.exists():
    raise FileNotFoundError(data_yaml)
config = yaml.safe_load(data_yaml.read_text(encoding="utf-8")) or {}
for required in ("images/train", "images/val", "labels/train", "labels/val"):
    if not (LOCAL_DATASET_DIR / required).exists():
        raise FileNotFoundError(LOCAL_DATASET_DIR / required)
if not config.get("names"):
    raise ValueError("data.yaml must define names")
print({"dataset_validation": "PASS", "classes": config["names"], "train_images": len(list((LOCAL_DATASET_DIR / "images/train").glob("*"))), "val_images": len(list((LOCAL_DATASET_DIR / "images/val").glob("*")))})

{'dataset_validation': 'PASS', 'classes': ['object'], 'train_images': 6, 'val_images': 2}


In [4]:
def tao_api(method, path, payload=None, headers=None):
    merged = {"Content-Type": "application/json"}
    if headers:
        merged.update(headers)
    response = http.request(method, TAO_BASE_URL + path, headers=merged, json=payload, timeout=(10, 120))
    response.raise_for_status()
    return response.json() if response.content else None

login = tao_api("POST", "/api/v2/login", {"ngc_key": NGC_KEY, "ngc_org_name": TAO_ORG, "enable_telemetry": False}, {"ngc_key": NGC_KEY})
AUTH = {"Authorization": f"Bearer {login['token']}"}
health = tao_api("GET", "/api/v2/health")
assert "readiness" in health
s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT, aws_access_key_id=S3_ACCESS_KEY, aws_secret_access_key=S3_SECRET_KEY, region_name="us-east-1")
uploaded = 0
for local_file in LOCAL_DATASET_DIR.rglob("*"):
    if local_file.is_file():
        key = f"{S3_PREFIX}/{local_file.relative_to(LOCAL_DATASET_DIR).as_posix()}"
        print(local_file)
        s3.upload_file(str(local_file), S3_BUCKET, key)
        uploaded += 1
DATASET_URI = f"seaweedfs://{S3_BUCKET}/{S3_PREFIX}"
print({"upload": "PASS", "objects": uploaded, "dataset_uri": DATASET_URI})

/tmp/yolo26-smoke/data.yaml


/tmp/yolo26-smoke/images/train/sample_004.png


/tmp/yolo26-smoke/images/train/sample_000.png


/tmp/yolo26-smoke/images/train/sample_005.png
/tmp/yolo26-smoke/images/train/sample_002.png


/tmp/yolo26-smoke/images/train/sample_001.png


...

/tmp/yolo26-smoke/labels/train/sample_004.txt


/tmp/yolo26-smoke/labels/train/sample_005.txt


/tmp/yolo26-smoke/labels/train/sample_002.txt
/tmp/yolo26-smoke/labels/train/sample_001.txt


/tmp/yolo26-smoke/labels/val/sample_000.txt


/tmp/yolo26-smoke/labels/val/sample_001.txt
{'upload': 'PASS', 'objects': 17, 'dataset_uri': 'seaweedfs://tao-storage/data/yolo26-scientist-5b8590d5'}


In [5]:
existing = tao_api("GET", f"/api/v2/orgs/{TAO_ORG}/datasets?name={DATASET_NAME}", headers=AUTH)
items = existing.get("datasets", []) if isinstance(existing, dict) else existing
if items:
    tao_dataset = items[0]
else:
    tao_dataset = tao_api("POST", f"/api/v2/orgs/{TAO_ORG}/datasets", {"name": DATASET_NAME, "shared": False, "type": "object_detection", "format": "coco", "workspace": WORKSPACE_ID, "cloud_file_path": S3_PREFIX, "use_for": ["training", "evaluation", "testing"], "skip_validation": True}, AUTH)
TAO_DATASET_ID = tao_dataset.get("id")
print({"tao_dataset_id": TAO_DATASET_ID, "dataset_uri": DATASET_URI})

{'tao_dataset_id': '753b27ca-cfca-4d0e-993e-863d0984fb82', 'dataset_uri': 'seaweedfs://tao-storage/data/yolo26-scientist-5b8590d5'}


In [6]:
def adapter(method, path, payload=None):
    last_error = None
    for attempt in range(8):
        try:
            response = http.request(method, ADAPTER_URL + path, json=payload, timeout=(10, 120))
            response.raise_for_status()
            return response.json()
        except requests.RequestException as error:
            last_error = error
            if method != "GET" or attempt == 7:
                raise
            time.sleep(min(2 ** attempt, 20))
    raise last_error

def wait_adapter(job_id):
    deadline = time.monotonic() + TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        current = adapter("GET", f"/v1/jobs/{job_id}")
        print(job_id, current.get("action"), current.get("status"))
        if current.get("status") in {"done", "error"}:
            if current["status"] != "done":
                raise RuntimeError(json.dumps(current, indent=2))
            return current
        time.sleep(POLL_SECONDS)
    raise TimeoutError(job_id)

train_payload = {"model_name": DATASET_NAME, "architecture": "yolo26n", "dataset_uri": DATASET_URI, "tao_workspace_id": WORKSPACE_ID, "tao_dataset_id": TAO_DATASET_ID, "input_size": 640, "max_epochs": 1, "batch_size": 2, "patience": 20, "device": "0", "train": {"epochs": 1, "imgsz": 640, "batch": 2, "patience": 20, "device": "0", "workers": 1, "optimizer": "auto", "lr0": 0.01, "weight_decay": 0.0005, "amp": True, "cache": False, "seed": 42}}
if not RUN_EXPERIMENT:
    print("dry-run: upload e registro executados; nenhum treino submetido")
else:
    train = adapter("POST", "/v1/jobs/train", train_payload)
    train_result = wait_adapter(train["job_id"])
    evaluate = adapter("POST", f"/v1/jobs/{train['job_id']}/evaluate", {"dataset_uri": DATASET_URI, "input_size": 640, "device": "0"})
    evaluate_result = wait_adapter(evaluate["job_id"])
    export = adapter("POST", f"/v1/jobs/{train['job_id']}/export", {"input_size": 640, "device": "0"})
    export_result = wait_adapter(export["job_id"])
    print(json.dumps({"train": train_result, "evaluate": evaluate_result, "export": export_result}, indent=2, default=str))

7150c4c0-c87b-4c50-a65c-98590af6aa16 train queued


7150c4c0-c87b-4c50-a65c-98590af6aa16 train running


7150c4c0-c87b-4c50-a65c-98590af6aa16 train done


eaef76ff-e364-475d-97e6-2fd6d8482b78 evaluate running


eaef76ff-e364-475d-97e6-2fd6d8482b78 evaluate done


4cf705e2-bf3f-4cb1-b396-ddd2631b4f86 export running


4cf705e2-bf3f-4cb1-b396-ddd2631b4f86 export done
{
  "train": {
    "id": "7150c4c0-c87b-4c50-a65c-98590af6aa16",
    "action": "train",
    "status": "done",
    "model_name": "yolo26-scientist-5b8590d5",
    "architecture": "yolo26n",
    "dataset_path": null,
    "dataset_uri": "seaweedfs://tao-storage/data/yolo26-scientist-5b8590d5",
    "tao_workspace_id": "98c4bec1-88df-423e-8a15-af78881e9742",
    "tao_dataset_id": "753b27ca-cfca-4d0e-993e-863d0984fb82",
    "input_size": 640,
    "max_epochs": 1,
    "batch_size": 2,
    "patience": 20,
    "device": "0",
    "train": {
      "epochs": 1,
      "imgsz": 640,
      "batch": 2,
      "patience": 20,
      "device": "0",
      "workers": 1,
      "optimizer": "auto",
      "lr0": 0.01,
      "weight_decay": 0.0005,
      "amp": true,
      "cache": false,
      "seed": 42
    },
    "data_yaml": "/shared-storage/ultralytics-jobs/7150c4c0-c87b-4c50-a65c-98590af6aa16/data.yaml",
    "version": "v1",
    "artifact_path": "/shared-sto